# CR-2 G2s 探す段（Claude バッチ API・CPU）

段3 で固定したリクエストから、計画の定義 17 のとおりに**探す段**を回します。全部終わったら**段3b の記録**（問題ごとの窓・比べ用の単語・確かめる段のリクエストの一覧のハッシュ）を Drive に書いて**止まります**。確かめる段（G2c）は別のノートブックです。

- **送る前に（G2 の前の受け入れ）**：段2・G1 の結果から 1 回目のリクエストの一覧を作り直し、段3 の固定のハッシュと Drive の一覧の両方と一致することを確かめます。違えば 1 件も送りません。
- **回し方**：1 回目は全部の問題の格子 5 か所 × 32 本。2 回目からは、まだ止まっていない問題の真ん中の 1 か所 × 32 本（最大 5 回）。各回が全部そろってから、コードが次の切れ目を決めます。32 本中 28 本未満しか戻らなかった切れ目があれば、その問題は外れます（計画の決まり）。
- **お金**：バッチを出す前に毎回 count_tokens で数え、「使った額＋流れているバッチの最悪の額＋このバッチの最悪の額」が **$54** を超えるなら出しません。流れているバッチがないとき、**確かめる段の上限まで含めた**残りの見込みが足りなければ止めます（`INCOMPLETE`）。$60 を超えない保証ではありません。
- 途中で切れても、セル1 → セル2 → セル3 で続きから回ります。ランタイムは 1 つだけ。
- 問題文・応答は表示しません。応答は Drive の非公開ファイル（`*_PRIVATE.jsonl`）にだけ保存します。

**使い方**：CPU のランタイム。シークレットは `ANTHROPIC_API_KEY` と `GITHUB_TOKEN`。セル1 → セル2 → セル3。数時間〜半日かかることがあります。終わったら Claude に教えてね（Claude が段3b を Git に固定します）。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact head, ancestors, (pinned SDK and key, never printed), notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
EXPECTED_HEAD = "da200a4805bdd727f4bb4b4094a08aa070c31a47"
ANCESTORS = ["311e3008211ebaca0735abccf3501e6fcb8da3df", "da200a4805bdd727f4bb4b4094a08aa070c31a47"]
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
for anc in ["7949d74"] + ANCESTORS:
    assert subprocess.run(["git", "-C", str(REPO), "merge-base", "--is-ancestor", anc, head]).returncode == 0, anc + " が祖先ではありません"
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from pork import cr2 as C
from pork import cr2_search as S
from pork import cr2_run as RN
from pork import cr2_g2 as G
from pork import cr2_g3 as G3
from pork import cr2_stage3b as T3
from pork import cr1_run as R1
from pork import alt_prelook as P
import numpy
RN.check_prompt()
D = Path("/content/drive/MyDrive/DECISION_POINTS/cr2")
D.mkdir(parents=True, exist_ok=True)
JST = dt.timezone(dt.timedelta(hours=9))

def git_blob(path):
    return subprocess.check_output(["git", "hash-object", path], text=True).strip()

def write_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    R1._atomic_write(path, obj)
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_chase_verified():
    """Plan 1. (H4, I3): pinned revision, canonical hash from the Hugging Face API vs the downloaded bytes."""
    from huggingface_hub import HfApi, hf_hub_download
    info = HfApi().dataset_info(C.DATASET_ID, revision=C.DATASET_REVISION, files_metadata=True, token=False)
    sib = next((s for s in info.siblings if s.rfilename == C.DATASET_FILE), None)
    meta = None
    if sib is not None:
        lfs = sib.lfs
        if lfs is not None and not isinstance(lfs, dict):
            lfs = {"sha256": getattr(lfs, "sha256", None), "size": getattr(lfs, "size", None)}
        meta = {"blob_id": sib.blob_id, "lfs": lfs}
    p = hf_hub_download(C.DATASET_ID, C.DATASET_FILE, repo_type="dataset", revision=C.DATASET_REVISION, token=False)
    data = Path(p).read_bytes()
    prov = C.check_provenance(data, meta, info.sha)
    rp = hf_hub_download(C.DATASET_ID, "README.md", repo_type="dataset", revision=C.DATASET_REVISION, token=False)
    readme = Path(rp).read_text(encoding="utf-8")
    rows = C.parse_rows(data)
    del data
    return rows, prov, {"readme_sha256": C.sha256_text(readme), "license": C.readme_license(readme)}

def rebuild_stage2(file_sha=None, cand_sha=None, excl_sha=None):
    rows, prov, readme = load_chase_verified()
    if file_sha is not None:
        assert prov["file_sha256"] == file_sha, "データのファイルが段2 の固定と違います。止めます（1 件も送りません）。"
    pre = C.reproduce_prelook3(rows, P.load_math())
    cand = C.build_candidates(rows, pre)
    if cand_sha is not None:
        assert cand["candidates_sha256"] == cand_sha, "候補の一覧が段2 の固定と違います。止めます。"
    if excl_sha is not None:
        assert cand["excluded_sha256"] == excl_sha, "除外した行が段2 の固定と違います。止めます。"
    by_id = {int(r["ID"]): r for r in rows}
    cands = [{**c, "question": C.problem_text(by_id[c["id"]]), "gold": str(C.gold(by_id[c["id"]]))}
             for c in cand["candidates"]]                                     # PRIVATE, memory only
    del rows, by_id
    return cand, cands, prov, readme, pre

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic==" + RN.SDK_VERSION], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
import anthropic
assert anthropic.__version__ == RN.SDK_VERSION, anthropic.__version__
_key = get_secret("ANTHROPIC_API_KEY")
assert _key, "Colab のシークレットに ANTHROPIC_API_KEY を登録してね。"
client = anthropic.Anthropic(api_key=_key)                       # reads, count_tokens: default retries
create_client = anthropic.Anthropic(api_key=_key, max_retries=0)  # sends: never retried by the SDK
del _key
TRANSIENT = (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.InternalServerError, anthropic.APITimeoutError)
print("anthropic", anthropic.__version__)

print("study:", C.STUDY, "| exact head:", head, "| numpy", numpy.__version__, "| python", sys.version.split()[0])

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'CR-2 G2s 探す段（バッチAPI・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - CR-2 G2s search', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — G2 の前の受け入れ.
# 前のランタイムが止まっていることを千紗さんが確かめたときだけ、ここにその番号を入れる（ふだんは空のまま）
OWNER_CONFIRMS_OLD_RUNTIME_STOPPED = ""
G1_HEAD = "311e3008211ebaca0735abccf3501e6fcb8da3df"
CANDIDATES_SHA = "c4bffbfa9632ed51a1222fae0bc65b256b3db0ae7e1b0d12e8bf70ac649f2339"
FILE_SHA = "731f2db9a590ac6c49ab7529d08dee0fcb09cc832a6462248e6939919319222c"
FINAL_ITEMS_SHA = "9ffd511cff0024e5106e50d18b5559f179485d10d528c973a2f458648ab8fee9"
ROUND0_SHA = "7f87bc608aaa03b2961c7a17ae758ccb82c16d5a7841b4b29e1a53dbc6706aae"

assert subprocess.run(["git", "diff", "--quiet", G1_HEAD, head, "--", "pork", "scripts", "tests"]).returncode == 0, \
    "G1 のあとでコードが変わっています。止めます（1 件も送りません）。"
LOCK3 = json.loads((REPO / "notes/cr2_lock_stage3.json").read_text())                  # the committed 段3 lock
assert LOCK3["code_blobs"] == {"pork/cr2_search.py": git_blob("pork/cr2_search.py"), "pork/cr2.py": git_blob("pork/cr2.py")}, "定義 17・18 のコードが段3 の固定と違います。止めます。"
assert LOCK3["numpy"] == numpy.__version__, "numpy の版が段3 の固定と違います。止めます（Claude に教えてね）。"
assert LOCK3["sdk"] == RN.SDK_VERSION and LOCK3["price_usd_per_mtok"] == {"in": RN.PRICE_IN, "out": RN.PRICE_OUT, "batch_factor": 0.5}
assert LOCK3["budget"]["B_stop"] == C.B_STOP and LOCK3["budget"]["B_cap"] == C.B_CAP and LOCK3["budget"]["worst_output_tokens"] == C.MAX_TOKENS
assert LOCK3["final_items_sha256"] == FINAL_ITEMS_SHA and LOCK3["file_sha256"] == FILE_SHA
assert LOCK3["rules"] == {"grid": S.GRID, "stop_width": S.STOP_WIDTH, "K_S": S.K_S, "min_returned": S.MIN_RETURNED,
                          "controls": S.N_CONTROLS, "K": C.K, "K_A": C.K_A, "SEED_CR2_WORD": C.SEED_CR2_WORD,
                          "control_key": "default_rng([SEED_CR2_WORD, ID])"}, "探す段・確かめる段の決まりが段3 の固定と違います。止めます。"

assert LOCK3["search_round0_manifest_sha256"] == ROUND0_SHA

cand, cands, prov, readme, pre = rebuild_stage2(FILE_SHA, CANDIDATES_SHA)
P_ = RN.paths()
ck = json.loads((Path("/content/drive/MyDrive") / P_["checkpoint"]).read_text())
assert ck["header"] == RN.header(G1_HEAD, cand["candidates_sha256"], CANDIDATES_SHA, RN.SDK_VERSION, C.DATASET_REVISION, FILE_SHA, FILE_SHA)
RN.validate(ck, cands)
g0 = json.loads((D / "G0/CR2_G0.json").read_text())
assert g0["pass"] and g0["code_head"] == G1_HEAD
g1 = RN.summarize_g1(ck, cands, g0["spent_usd"])
assert g1["verdict"] == "GO" and g1["final_items_sha256"] == FINAL_ITEMS_SHA, "G1 の結果が段3 の固定と違います。止めます。"
questions = {C.cid(c["id"]): c["question"] for c in cands}
ITEMS = RN.manifest_items(g1, ck, questions)
PRIOR = g0["spent_usd"] + g1["g1_spent_usd"]
PER_G1B = g1["G1b"]["estimate"]["per_request_usd_base"]      # early_stop applies the x1.2
del questions, cands

dm = json.loads((D / "G2S/CR2_G2S_round0_manifest.json").read_text())
assert dm["manifest_sha256"] == ROUND0_SHA == C.rows_hash(dm["rows"]), "Drive の 1 回目の一覧が段3 の固定と違います。止めます。"
rows0, sha0 = RN.search_round0(ITEMS)
assert sha0 == ROUND0_SHA, "作り直した 1 回目の一覧が段3 の固定と違います。止めます。"
del dm, rows0
OUT = D / "G2S"
R1.drive_smoke(OUT)

from pork import cr1_g2 as G1x
API = G1x.AnthropicBatchAPI(client, create_client)
import uuid
_rid = Path("/content/g2_runtime_id")
if not _rid.exists():
    _rid.write_text(uuid.uuid4().hex[:12])
RUNTIME = _rid.read_text().strip()
print("このランタイムの番号:", RUNTIME)

def run_list(ST, REQ, save, fence, label):
    last = None
    while True:
        try:
            fence()
            G.claim_lease(ST, RUNTIME, dt.datetime.now(JST), save)
            G.collect(ST, REQ, API, OUT, save)
            if ST["submitting"]:
                G.resolve_submitting(ST, API, save, dt.datetime.now(JST))
            G.submit_ready(ST, REQ, API, save, dt.datetime.now(JST), API.count_tokens, fence_check=fence)
        except TRANSIENT as e:
            print("一時的なエラー:", type(e).__name__, "→ 1 分後にもう一度")
            time.sleep(60)
            continue
        save()
        if ST["stopped"] or G.done(ST, REQ):
            break
        t = G.tallies(ST, REQ)
        line = (f"{label} | 成功 {len(t['succeeded'])}/{len(REQ)} | 流れている {len(t['inflight'])} | 残りのかたまり {len(t['chunks_left'])}"
                f" | 出し直し待ち {len(t['retry'])} | 確定した額 ${G.spent_usd(ST):.2f}")
        if line != last:
            print(dt.datetime.now(JST).strftime("%H:%M"), line)
            last = line
        time.sleep(60)
    G.verify_collected(ST, REQ, OUT, api=API, save=save)

def open_list(path, HDR, REQ):
    if path.exists():
        saved = json.loads(path.read_text())
        if saved["header"]["code_head"] != head:
            raise G.G2Stop("the state was written by another code commit; ask Claude before resuming")
        ST = G.open_state(saved, HDR)
    else:
        ST = G.open_state(None, HDR)
    save = lambda: G.save_state(path, ST)
    fence = lambda: G.fence(ST, lambda: json.loads(path.read_text()))
    G.claim_lease(ST, RUNTIME, dt.datetime.now(JST), save, owner_confirms_stopped=OWNER_CONFIRMS_OLD_RUNTIME_STOPPED or None)
    G.verify_collected(ST, REQ, OUT, api=API, save=save)
    G.resolve_submitting(ST, API, save, dt.datetime.now(JST))
    save()
    return ST, save, fence

print("G2 の前の受け入れ: OK | 問題", len(ITEMS), "| 1 回目のリクエスト", len(ITEMS) * 5 * 32)

In [ ]:
# Cell 3 — the search rounds, then the 段3b record (Drive) and stop.
ROUND_STATES, ROUND_RESULTS, MANIFESTS = [], [], []
stopped = None
for r in range(T3.MAX_ROUNDS + 1):
    plan = T3.plan_search(ITEMS, ROUND_RESULTS)
    if plan["complete"]:
        break
    cuts = plan["next_cuts"]
    rows, sha = C.manifest_rows(ITEMS, cuts, "s")
    if r == 0:
        assert sha == ROUND0_SHA
    REQ = G.Requests(ITEMS, cuts, "s", sha)
    act = G.mean_actual_per_request(ROUND_STATES)
    HDR = G.g2_header(sha, head, anthropic.__version__, PRIOR + G.stage_spent(ROUND_STATES), act if act is not None else PER_G1B,
                      "s", r, T3.future_requests(ITEMS, plan, cuts))
    ST, save, fence = open_list(OUT / f"CR2_G2S_round{r}_state.json", HDR, REQ)
    print(f"探す段 {r + 1} 回目: リクエスト {len(REQ)}（問題 {len(cuts)}）")
    run_list(ST, REQ, save, fence, f"探す段 {r + 1} 回目")
    write_json(OUT / f"CR2_G2S_round{r}_summary.json", G.summary(ST, REQ))
    if ST["stopped"]:
        stopped = ST["stopped"]
        break
    ROUND_STATES.append(ST)
    ROUND_RESULTS.append(G.per_item_results(ST, REQ, OUT))
    MANIFESTS.append({"manifest_sha256": sha, "requests": len(REQ)})
if stopped is None:
    plan = T3.plan_search(ITEMS, ROUND_RESULTS)
    rec = T3.stage3b_record(ITEMS, plan, MANIFESTS, FINAL_ITEMS_SHA)
    txt = T3.canonical(rec)
    p3b = D / "STAGE3B/CR2_stage3b_record.json"
    p3b.parent.mkdir(parents=True, exist_ok=True)
    tmp = p3b.with_name(p3b.name + ".tmp"); tmp.write_text(txt); os.replace(tmp, p3b)
    h3b = T3.stage3b_sha256(rec)
    s = {"schema": "cr2-g2s-summary-v1", "code_head": head, "rounds": len(MANIFESTS), "items": len(ITEMS),
         "windows": len(rec["items"]), "dropped": len(rec["dropped"]), "confirmation_requests": rec["confirmation_requests"],
         "stage3b_sha256": h3b, "spent_total_usd": PRIOR + G.stage_spent(ROUND_STATES),
         "g2s_spent_usd": G.stage_spent(ROUND_STATES)}
    write_json(OUT / "CR2_G2S_summary.json", s)
    print("探す段: 完了 |", s["rounds"], "回 | 窓", s["windows"], "| 外れた", s["dropped"], "| 確かめる段のリクエスト",
          s["confirmation_requests"], "| 合計 $%.2f" % s["spent_total_usd"])
    print("stage3b_sha256:", h3b)
    msg = '終わりました：CR-2 の探す段が終わり、段3b の記録を Drive に書きました（Claude に固定してもらってね）'
else:
    print("止まりました:", stopped)
    rep = G3.incomplete_report("G2s", {"round": len(MANIFESTS), "stopped": stopped,
                                       "spent_total_usd": PRIOR + G.stage_spent(ROUND_STATES) + G.spent_usd(ST) - float(ST["header"]["prior_spent_usd"])},
                               len(ITEMS))
    write_json(D / "G3/CR2_G3_result.json", json.loads(G3.to_json(rep)))
    msg = '止まりました：計画の決まりで探す段を止めました（G3 の出し先に INCOMPLETE を書きました。Claude に見せてね）'
notify(msg, tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()